In [1]:
!nvidia-smi

Tue Sep  2 09:03:49 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 511.65       Driver Version: 511.65       CUDA Version: 11.6     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0  On |                  N/A |
| N/A   64C    P0   129W /  N/A |   5777MiB / 16384MiB |    100%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
import numpy as np
import torch
from sklearn.datasets import make_blobs
n_dim = 2
tr_x, tr_y = make_blobs(n_samples=80, n_features=2, centers=[[1,1], [-1,-1], [1,-1], [-1,1]], shuffle=True, cluster_std=.3)
tt_x, tt_y = make_blobs(n_samples=20, n_features=2, centers=[[1,1], [-1,-1], [1,-1], [-1,1]], shuffle=True, cluster_std=.3)
ck_y = tr_y.copy()
def label_map(y, from_, n):
    ck_y = y.copy()
    for i in from_:
        ck_y[y==i] = n
    return ck_y
# tr_y=label_map(tr_y, [0,1], 0)
# tr_y=label_map(tr_y, [2,3], 1)
# tt_y=label_map(tt_y, [0,1], 0)
# tt_y=label_map(tt_y, [2,3], 1)
t_tr_x = torch.FloatTensor(tr_x)
t_tt_x = torch.FloatTensor(tt_x)
t_tr_y = torch.FloatTensor(tr_y)
t_tt_y = torch.FloatTensor(tt_y)
class NeuralNet(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__() # 부모 생성자 동작
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.l1 = torch.nn.Linear(self.input_size, self.hidden_size)
        self.relu = torch.nn.ReLU()
        self.l2 = torch.nn.Linear(self.hidden_size, 1) # 여기서는 어차피 출력이 한개이므로..
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, input_tensor):
        l1 = self.l1(input_tensor)
        relu = self.relu(l1)
        l2 = self.l2(relu)
        output = self.sigmoid(l2)
        return output
m = NeuralNet(2,5)
criterion = torch.nn.BCELoss()
opt = torch.optim.SGD(m.parameters(), lr=3e-2)
epochs = 1000
for epoch in range(epochs): # 학습 시엔 train과 backward가 반드시 와야 함
    m.train()
    opt.zero_grad() # 경사 정리
    tr_output = m(t_tr_x)
    tr_loss = criterion(tr_output.squeeze(), t_tr_y)
    if epoch % 100 == 0:
        print(f'epoch:{epoch}\t loss: {tr_loss.item()}')
    tr_loss.backward() # 가중치 갱신(역전파)
    opt.step()
m.eval()
py = m(t_tt_x)
ty = t_tt_y
test_loss = criterion(py.squeeze(), ty)
test_loss.item()

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils import data
from torchvision import datasets, transforms, utils

In [2]:
transform = transforms.Compose([transforms.ToTensor()])
datasets

<module 'torchvision.datasets' from 'c:\\Users\\devchoi\\miniconda3\\Lib\\site-packages\\torchvision\\datasets\\__init__.py'>

In [3]:
tr_ds = datasets.FashionMNIST(root='./data/', train=True, download=True, transform=transform)
tt_ds = datasets.FashionMNIST(root='./data/', train=False, download=True, transform=transform)

100%|██████████| 26.4M/26.4M [00:04<00:00, 5.62MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 103kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 1.80MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 28.4MB/s]


In [4]:
tr_ds_loader = data.DataLoader(dataset=tr_ds, batch_size=64)
tt_ds_loader = data.DataLoader(dataset=tt_ds, batch_size=64)

In [5]:
ds = iter(tr_ds_loader)
img, label = next(ds)

In [6]:
img.shape, label.shape, label

(torch.Size([64, 1, 28, 28]),
 torch.Size([64]),
 tensor([9, 0, 0, 3, 0, 2, 7, 2, 5, 5, 0, 9, 5, 5, 7, 9, 1, 0, 6, 4, 3, 1, 4, 8,
         4, 3, 0, 2, 4, 4, 5, 3, 6, 6, 0, 8, 5, 2, 1, 6, 6, 7, 9, 5, 9, 2, 7, 3,
         0, 3, 3, 3, 7, 2, 2, 6, 6, 8, 3, 3, 5, 0, 5, 5]))

In [ ]:
labels_map = {
    0: "티셔츠/탑 t-shirt/top",
    1: "트라우저 trouser",
    2: "풀오버 pullover",
    3: "드레스 dress",
    4: "코트 coat",
    5: "샌들 sandal",
    6: "셔츠 shirt",
    7: "스니커즈 sneaker",
    8: "가방 bag",
    9: "앵클 부츠 ankle boot",
}

In [9]:
idx = label[0].item()
labels_map[idx]

'앵클 부츠 ankle boot'

In [11]:
# 기본적으로 가져오는 것들이라고 보면 됨
from torchvision import transforms, datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [17]:
USE_CUDA = torch.cuda.is_available() # cuda를 사용중인지
DEVICE = torch.device('cuda' if USE_CUDA else 'cpu')
BATCH_SIZE = 64 # 작업 준비
DEVICE

device(type='cuda')

In [18]:
# 1. 데이터 준비 torch.shape는(c,h,w)
transform = transforms.Compose([
    transforms.ToTensor()
]) # 데이터 사용 방식 내용 결정
tr_ds = datasets.FashionMNIST(
    root='./data/',
    train=True,
    download=False,
    transform=transform
)
tt_ds = datasets.FashionMNIST(
    root='./data/',
    train=False,
    download=False,
    transform=transform
)

tr_ds_loader = data.DataLoader(dataset=tr_ds, batch_size=BATCH_SIZE, shuffle=True)
tt_ds_loader = data.DataLoader(dataset=tt_ds, batch_size=BATCH_SIZE, shuffle=True)

In [21]:
class DNN(nn.Module):
    def __init__(self): # 클래스를 재사용하기 위해 이런 식으로 설계함
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self,x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [28]:
m = DNN().to(DEVICE)
opt = optim.SGD(m.parameters(), lr=1e-2)

In [24]:
# 학습함수 설계
def train(m, tr_ds_loader, opt):
    m.train()
    for i, (data, target) in enumerate(tr_ds_loader):
        data, target = data.to(DEVICE), target.to(DEVICE)
        opt.zero_grad()
        output = m(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        opt.step()

In [25]:
def evaluate(m, tt_ds_loader):
    m.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad(): # 회차별로 메모리를 사용하여 쓰고 지우고를 반복하여 메모리가 계속 적재되는 것을 방지 + 기울기값 미사용
        for data, target in tt_ds_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = m(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item() # 데이터 구조를 일치하도록 만들기 위해 하는 작업
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct/len(tt_ds_loader.dataset)*100.
    return test_loss, test_accuracy

In [29]:
EPOCHS = 30
for i in range(1, EPOCHS+1):
    train(m, tr_ds_loader, opt)
    test_loss, test_acc = evaluate(m, tt_ds_loader)
    print(f'epochs: {i}\ttest_loss: {test_loss}\ttest_accuracy: {test_acc}')

epochs: 1	test_loss: 0.8371625259399414	test_accuracy: 68.36
epochs: 2	test_loss: 0.6556618327140808	test_accuracy: 76.61
epochs: 3	test_loss: 0.6093950707912446	test_accuracy: 78.12
epochs: 4	test_loss: 0.534598556804657	test_accuracy: 81.08
epochs: 5	test_loss: 0.5227257287979126	test_accuracy: 81.11
epochs: 6	test_loss: 0.5153257967948913	test_accuracy: 81.27
epochs: 7	test_loss: 0.4837731725692749	test_accuracy: 82.72
epochs: 8	test_loss: 0.4651974112510681	test_accuracy: 83.28999999999999
epochs: 9	test_loss: 0.4713092756271362	test_accuracy: 82.82000000000001
epochs: 10	test_loss: 0.46068357751369476	test_accuracy: 83.34
epochs: 11	test_loss: 0.4503320085525513	test_accuracy: 83.7
epochs: 12	test_loss: 0.44351040210723874	test_accuracy: 84.0
epochs: 13	test_loss: 0.4484978870868683	test_accuracy: 84.25
epochs: 14	test_loss: 0.47969695196151735	test_accuracy: 82.67999999999999
epochs: 15	test_loss: 0.475537556886673	test_accuracy: 82.59
epochs: 16	test_loss: 0.42474263916015625	te

In [ ]:
m.eval()
with torch.no_grad(): # 속도와 메모리 사용량의 개선을 하게 됨(기울기 계산을 안하고 시각화를 안하게 되서 그러함)
    m(data)

# 데이터를 로드하여 모델 층을 완전연결층 구조로 4층 쌓은 DNN 구조를 설계하고 학습후 모델을 예측 및 검증하시오

In [33]:
BATCH_SIZE = 128
EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [34]:
transform = transforms.Compose([transforms.ToTensor()])
tr_ds = datasets.FashionMNIST(root='./data/', train=True, download=False, transform=transform)
tt_ds = datasets.FashionMNIST(root='./data/', train=False, download=False, transform=transform)

In [35]:
tr_ds_loader = data.DataLoader(dataset=tr_ds, batch_size=BATCH_SIZE)
tt_ds_loader = data.DataLoader(dataset=tr_ds, batch_size=BATCH_SIZE)

In [36]:
class DNN2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self,x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [42]:
m2 = DNN2().to(DEVICE)
opt = optim.Adam(m2.parameters())

In [38]:
def train(m2, tr_ds_loader, opt):
    m2.train()
    for data, target in tr_ds_loader:
        opt.zero_grad()
        data, target = data.to(DEVICE), target.to(DEVICE)
        output = m2(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        opt.step()

In [39]:
def eval(m2, tt_ds_loader):
    m2.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for data, target in tt_ds_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = m2(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item() # loss값만
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset) * 100
    return test_loss, test_accuracy


In [43]:
for i in range(1, EPOCHS+1):
    train(m2, tr_ds_loader, opt)
    test_loss, test_acc = eval(m2, tt_ds_loader)
    print(f'epochs:{i}\tloss:{test_loss:.10f}\taccuracy:{test_acc:.5f}')

epochs:1	loss:0.4009746924	accuracy:85.50833
epochs:2	loss:0.3472274208	accuracy:87.30167
epochs:3	loss:0.3207084429	accuracy:88.19667
epochs:4	loss:0.2987712745	accuracy:88.81500
epochs:5	loss:0.2858449726	accuracy:89.28500
epochs:6	loss:0.2610748777	accuracy:90.22833
epochs:7	loss:0.2568613496	accuracy:90.56833
epochs:8	loss:0.2373075019	accuracy:91.03167
epochs:9	loss:0.2444285901	accuracy:90.66167
epochs:10	loss:0.2335425406	accuracy:90.80500


In [ ]:
# 예측 및 검증
py = m2(data)
F.log_softmax(py)

In [32]:
# 0. 작업 준비(이미지 처리 관점)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, datasets
import numpy as np
import matplotlib.pyplot as plt

In [33]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 60000
EPOCHS = 10
# tr_ds_loader = data.DataLoader(dataset=tr_ds, batch_size=BATCH_SIZE, shuffle=False)
# img, _ = next(iter(tr_ds_loader))
# img.shape
# tt_ds_loader = data.DataLoader(dataset=tr_ds, batch_size=BATCH_SIZE, shuffle=False)
# # 평균과 표준편차 계산
# img.mean(), img.std()

In [34]:
# 데이터 수정(노이즈 삽입)
# 1. 데이터 준비
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), # 데이터 증강(노이즈 삽입)
    transforms.ToTensor(), # 입력 데이터 정리
    transforms.Normalize((0.2860,), (0.3530,)) # 데이터 정규화 (이 과정 이전에 텐서화가 되어야 함)
])
tr_ds_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(root='./data/', train=True, download=False, transform=transform),
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(root='./data/', train=False, download=False, transform=transform),
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [35]:
class DNNmodel(nn.Module):
    def __init__(self, input_n, hidden_ns, output_n, dropout_p=0.2):
        super().__init__()
        self.fc_in = nn.Linear(input_n, hidden_ns[0])
        # GPU에서 사용시 리스트도 nn모듈에 있는 기능으로 사용해야됨
        self.fc_h_l = nn.ModuleList([nn.Linear(hidden_ns[i], hidden_ns[i+1]) for i in range(len(hidden_ns)-1)])
        # self.fc_h_l = [nn.Linear(hidden_ns[i], hidden_ns[i+1]) for i in range(len(hidden_ns)-1)]
        self.fc_out = nn.Linear(hidden_ns[-1], output_n)
        
        self.dropout_p = dropout_p
        self.input_n = input_n
        self.hidden_ns = hidden_ns
        self.output_n = output_n

    def forward(self, x):
        x = x.view(-1, self.input_n) # 벡터화
        x = F.relu(self.fc_in(x)) # 입력계층 연산
        x = F.dropout(x, training=self.training, p=self.dropout_p)
        for i in range(len(self.fc_h_l)): # 은닉계층 연산
            x = F.relu(self.fc_h_l[i](x))
            x = F.dropout(x, training=self.training, p=self.dropout_p)
        output = self.fc_out(x) # 출력계층 연산
        return output
# # 위의 self.hidden_ns의 설명을 위한 코드
# x=0
# def f(x):
#     return x+1
# for i in [1,2,3]:
#     x = f(x)
#     print(f(x))

In [36]:
m = DNNmodel(784, [256, 128, 64], 10).to(DEVICE)
opt = optim.SGD(m.parameters(), lr=1e-2)

In [37]:
def train(m, tr_ds_loader, opt):
    m.train()
    for x,y in tr_ds_loader:
        data, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        py = m(data)
        loss = F.cross_entropy(py, target)
        loss.backward()
        opt.step()

In [38]:
def evaluate(m, tt_ds_loader):
    m.eval()
    correct, test_loss = 0, 0
    with torch.no_grad():
        for data, target in tt_ds_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = m(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item() # 손실값만 합산
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset) * 100
    return test_loss, test_accuracy

In [39]:
for i in range(1, EPOCHS+1):
    train(m, tr_ds_loader, opt)
    test_loss, test_acc = evaluate(m, tt_ds_loader)
    print(f'epochs:{i}\t loss:{test_loss}\t accuracy:{test_acc}')

epochs:1	 loss:2.3100306640625	 accuracy:9.99
epochs:2	 loss:2.3093615234375	 accuracy:9.94
epochs:3	 loss:2.30852734375	 accuracy:10.15
epochs:4	 loss:2.3080087890625	 accuracy:9.99
epochs:5	 loss:2.3073689453125	 accuracy:10.01
epochs:6	 loss:2.306716796875	 accuracy:10.05
epochs:7	 loss:2.3060083984375	 accuracy:10.17
epochs:8	 loss:2.3051728515625	 accuracy:10.25
epochs:9	 loss:2.3045234375	 accuracy:10.24
epochs:10	 loss:2.3038734375	 accuracy:10.459999999999999


In [71]:
torch.cuda.is_available()

True